In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import os

from tqdm import tqdm
import torch
import torch.nn.functional as F
import numpy as np
from torch_geometric.data import Data, separate, InMemoryDataset
from torch_geometric.utils import to_dense_batch, dense_to_sparse
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GATConv, GraphConv
from torch.utils.data import IterableDataset

from SpatialTemporalEncoder import SpatialTemporalEncoder
from MENDRTrainer import MENDRTrainer 
from encoder import ConvEncoder
from MENDRContextualizer import mATTContextualizer
from R2E import R2E
from transforms import RandomTemporalCrop


# Have to rename or else it conflicts with PyG imports
from types import SimpleNamespace as ns


In [5]:
def load_dataset(path):
    ecs = []
    eos = []

    for subject in tqdm(os.listdir(path)):
        ec = torch.load(os.path.join(path, subject, 'EC_geometric.pt'))
        eo = torch.load(os.path.join(path, subject, 'EO_geometric.pt'))

        ec_indices = len(ec[0].x)
        eo_indices = len(eo[0].x)


        # Break up collated data into individual samples
        # Note that separate is undocumented and I'm not sure what it precisely does
        # but it seems to work and satisfy our use case 
        for i in range(ec_indices):
            ecs.append(separate.separate(cls=ec[0].__class__, batch=ec[0], idx=i, slice_dict=ec[1], decrement=False))

        if eo[1] is None:
            eos.append(eo[0])
        else:
            for i in range(eo_indices):
                eos.append(separate.separate(cls=eo[0].__class__, batch=eo[0], idx=i, slice_dict=eo[1], decrement=False))
    dataset = ecs + eos
    flattened_dataset = []
    for data in dataset:
        data.edge_attr = data.edge_attr.flatten()[:, None]
        flattened_dataset.append(data)
    return dataset

class CustomDataset(InMemoryDataset):
    def __init__(self, listOfDataObjects):
        super().__init__()
        self.data, self.slices = self.collate(listOfDataObjects)
    
    def __len__(self):
        return len(self.slices)
    
    def __getitem__(self, idx):
        sample = self.get(idx)
        return sample

In [17]:
datasetList = load_dataset('/home/azureuser/mycontainer/SmallSubset')
print(len(datasetList))
dataset = CustomDataset(datasetList)
print(len(dataset))
print(len(dataset.data), len(dataset.slices))
loader = DataLoader(datasetList, batch_size=4, shuffle=True)

  0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           | 0/20 [00:00<?, ?it/s]/

80
5
5 5


In [19]:
print(dataset[69])

Data(x=[19, 15360], edge_index=[2, 361], edge_attr=[361, 1], y=[0], pos=[19, 3])


In [7]:
print(loader.dataset[0])
print(loader.dataset[0].edge_index.max(), loader.dataset[0].edge_index.min())
out = next(iter(loader))
print(out)
print(out.ptr)
print(torch.tensor(np.vstack(out.x)).shape, torch.tensor(np.vstack(out.x)).dtype)
print(len(out.edge_attr), out.edge_attr[0].shape, out.edge_attr[0].dtype)
print(torch.tensor(np.stack(out.edge_attr)).shape)
print(torch.flatten(torch.tensor(np.stack(out.edge_attr)), start_dim=0, end_dim=1).shape)

Data(x=[19, 15360], edge_index=[2, 361], edge_attr=[361, 1], y=[0], pos=[19, 3])
tensor(18) tensor(0)
DataBatch(x=[4], edge_index=[2, 1444], edge_attr=[4], y=[4], pos=[76, 3], batch=[76], ptr=[5])
tensor([ 0, 19, 38, 57, 76])
torch.Size([76, 15360]) torch.float64
4 (361, 1) float64
torch.Size([4, 361, 1])
torch.Size([1444, 1])


In [8]:
configs = ns(
	seq_len = 15360,
	pred_len = 15360,
	top_k = 5,
	d_model = 19,
	d_ff = 9,
	num_kernels = 3,
	heads = 4,
	dropout = 0.1,
)

model = SpatialTemporalEncoder(configs)

encoder = ConvEncoder(in_features=19, encoder_h=256, enc_width=(3, 2, 2), dropout=0., enc_downsample=(3, 2, 2))

contextualizer_config = ns(
		in_features=256,
		dropout=0.1,
		start_token=-5,
		position_encoder=25,
		epochs=4
	)

contextualizer = mATTContextualizer(contextualizer_config)
r2e = R2E(contextualizer_config)

trainer_config = ns(
	mask_span=6,
	multi_gpu=False,
	encoder_grad_frac=1,
	learning_rate=1e-3,
	l2_weight_decay=1e-5,
	mask_rate=0.1,
	temp=0.1,
	permuted_encodings=False,
	permuted_contexts=False,
	enc_feat_l2=1e-5,
	unmasked_negative_frac=0.1,
	num_negatives=10
)

trainer = MENDRTrainer(model, encoder, contextualizer, r2e, trainer_config)

No GPU detected: training and model execution will be performed on CPU.


In [10]:
# Slower learning rate for the encoder
trainer.set_optimizer(torch.optim.Adam(trainer.parameters()))
trainer.add_batch_transform(RandomTemporalCrop(max_crop_frac=0.05))

In [12]:
print(len(dataset))
print(dataset[0])

5
Data(x=[19, 15360], edge_index=[2, 361], edge_attr=[361, 1], y=[0], pos=[19, 3])


In [26]:
trainer.fit(training_dataset=datasetList, epochs=5, batch_size=8)

Loading data with 4 additional workers
Training on 10 samples


Epoch 0:   0%|                                                                                                                                                                   | 0/10 [00:00<?, ?it/s]

10


Epoch 1:   0%|                                                                                                                                                                   | 0/10 [00:00<?, ?it/s]

10


Epoch 1:  10%|████████████▌                                                                                                                 | 1/10 [02:28<22:17, 148.63s/it, Similarity=7.72, loss=4.11]


KeyboardInterrupt: 